The first thing we need is to import two key components from LangChain:

init_chat_model — from langchain.chat_models, used to connect to an LLM
PromptTemplate — from langchain_core.prompts, used to manage and structure our prompts

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

authentication and set up secure API key management.

In [2]:
import os
from dotenv import load_dotenv

# load_dotenv()
# os.environ["OPENAI_API_KEY"] = "your-key-here"
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

The core idea: never hardcode API keys directly in your code. If someone accesses your codebase, they'd have full access to your API credits.

Instead, the safe approach is:

Store keys in a .env file — a simple text file with entries like OPENAI_API_KEY=sk-proj-...
Use the dotenv library to load those variables into your environment at runtime
LangChain automatically picks up standard variable names like OPENAI_API_KEY when initializing models
Add .env to your .gitignore so it never gets committed to version control

In [3]:
prompt_template_str = """
Your task is to explain the concept of {concept} to me in a way that is:

1. Clear and intuitive
2. Concise (in under 100 words)
3. Tailored specifically to me and what I already know

Use the following information about me to personalize your explanations:

- Background: Aspiring Data Scientist in the Automotive Industry
- Professional Interests: Extracting structured information from engineering documents, data pipelines
- Personal Interests: Automotive technology, AI applications in engineering

The personalization should be subtle and natural. Avoid forced references to my background that don't genuinely enhance my understanding of the concept.
"""

Use LangChain'e PromptTemplate to formalize the content

PromptTemplate.from_template() converts your string into a reusable template object
.format() replaces {concept} with the actual value — in this case "Scaling Laws"
print(prompt) lets you see exactly what gets sent to the LLM

In [4]:
# Create a prompt template
prompt_template = PromptTemplate.from_template(prompt_template_str)

# Define the concept to explain
concept = "Scaling Laws"

# Format the prompt with the concept
prompt = prompt_template.format(concept=concept)

print(prompt)


Your task is to explain the concept of Scaling Laws to me in a way that is:

1. Clear and intuitive
2. Concise (in under 100 words)
3. Tailored specifically to me and what I already know

Use the following information about me to personalize your explanations:

- Background: Aspiring Data Scientist in the Automotive Industry
- Professional Interests: Extracting structured information from engineering documents, data pipelines
- Personal Interests: Automotive technology, AI applications in engineering

The personalization should be subtle and natural. Avoid forced references to my background that don't genuinely enhance my understanding of the concept.



calling the LLM!
init_chat_model() creates a model interface for gpt-4o-mini via OpenAI — this is your connection to the LLM
model.invoke() sends your formatted prompt and returns the response
response.text is the actual generated content

In [5]:
# Create a model interface
model = init_chat_model("gpt-4o-mini", model_provider="openai")

# Call the model with the prompt
response = model.invoke(prompt)

"""
or we can use LCEL for modern approach
chain = prompt_template | model
response = chain.invoke({"concept": "Scaling Laws"})
"""

# Print the generated content
print(response.text)

Scaling Laws describe how a system's performance changes as its size increases. In the context of data, they suggest that larger datasets or models often show exponentially better results, akin to how a car's performance improves with a more powerful engine. This principle is crucial in AI, where training on more data can enhance predictive capabilities. Understanding these laws can help you evaluate how data pipeline designs can scale efficiently to handle larger volumes of engineering documents, leading to better insights in automotive technology.


# Using Gradio and defing as a Function

Since building front-end UIs isn't our focus in this course, we need a lightweight way to test our applications visually. That's where Gradio comes in — a Python library by Hugging Face that lets you create simple web interfaces with just a few lines of code, no frontend experience required.

Before adding the Gradio code, we need to wrap our LLM-calling logic in a function. Gradio needs a function to call when the user submits input.

Here's what I'd like you to do:

Create a new function called generate_explanation() that takes concept as an argument
Move the prompt formatting and model calling code inside it
Have it return response.text instead of printing it
Keep prompt_template and model initialized outside the function
Remove the print statements and the hardcoded concept = "Scaling Laws" line

In [6]:

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

import os
from dotenv import load_dotenv

# load_dotenv()
# os.environ["OPENAI_API_KEY"] = "your-key-here"

prompt_template_str = """
  Your task is to explain the concept of {concept} to me in a way that is:

  1. Clear and intuitive
  2. Concise (in under 100 words)
  3. Tailored specifically to me and what I already know

  Use the following information about me to personalize your explanations:

  - Background: Aspiring Data Scientist in the Automotive Industry
  - Professional Interests: Extracting structured information from engineering documents, data pipelines
  - Personal Interests: Automotive technology, AI applications in engineering

  The personalization should be subtle and natural. Avoid forced references to my background that don't genuinely enhance my understanding of the concept.
  """
# Create a prompt template
prompt_template = PromptTemplate.from_template(prompt_template_str)

# Create a model interface
model = init_chat_model("gpt-4o-mini", model_provider="openai")

def generate_explanation(concept):
  # Call the model with the prompt
  prompt = prompt_template.format(concept = concept)
  response = model.invoke(prompt)
  # Print the generated content
  return response.text

In [7]:
! pip install gradio


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
#add the Gradio interface
import gradio as gr

demo = gr.Interface(
    fn=generate_explanation,
    inputs=[gr.Textbox(label="Concept", lines=1)],
    outputs=[gr.Textbox(label="Explanation", lines=5)],
    flagging_mode="never",
    title="Concept Explainer",
    description="Get a concise, personalized explanation of any concept"
)

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


fn=generate_explanation tells Gradio which function to call when the user submits input
inputs and outputs define the text fields the user sees
demo.launch() is commented out since Optima handles the rendering — it's there for your own projects